# 02 Correction Validation

This notebook validates the RPF correction models used in the article. It compares the trainable `m8_xgb` model with the deterministic `m7_dtr` benchmark.

**Inputs.** It reads final Alpha and Beta parquet datasets from `dataset/final/` and the v2 config from `config/experiment_config.yaml`.

**Experiment design.** Alpha now uses complete leave-one-station-out validation across all 10 Alpha sites. Beta is evaluated with one Alpha-trained transfer model, one Beta prediction pass per method, and downstream per-site metric aggregation for all 8 Beta sites.

**Outputs.** The notebook writes prediction audit CSVs, correction metrics, confusion matrices, compact tables, and journal-ready figures under the Notebook 2 output folders. The Alpha and Beta site boxplots are both derived from the same prediction frames used for the aggregate metrics.

**Safety.** Keep `execution.run_full_correction_validation: false` for smoke/layout previews. Set it to `true` only when you are ready to run the real `m8_xgb` training and prediction workflow.

## 1. Imports And Paths

This section resolves the article root, imports the shared helper module, loads the config, and prints the run mode. The helper module contains the heavy experiment logic so the notebook remains readable, but every major helper call below explains what it does internally.


In [1]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

print(f"Full correction validation: {cfg['execution']['run_full_correction_validation']}")
print(f"Reuse correction prediction files: {cfg['execution'].get('reuse_correction_prediction_files', False)}")


Article root: c:\Users\samha\Documents\PyNRPF\publication\2_journal_article
Config schema: journal_v2
Output root: c:\Users\samha\Documents\PyNRPF\publication\2_journal_article\outputs
Full correction validation: True
Reuse correction prediction files: False


## 2. Preflight Readiness Check

`h.correction_validation_preflight()` performs a no-training readiness check. It loads the final datasets, confirms dependency availability, recomputes the complete Alpha LOSO site order, ranks all Beta sites by current RPF labels, and reports the expected full-run workload.

For the full run, expect 10 Alpha `m8_xgb` trainings/predictions, one Alpha-to-Beta `m8_xgb` training/prediction, two batched `m7_dtr` inference passes, 44 primary metric rows, 40 Alpha site rows, and 32 Beta site rows. This is the safest cell to run before switching the config into full mode.

In [2]:
# This helper does not train models or write experiment outputs.
preflight = h.correction_validation_preflight(article_root)
display(preflight["readiness"])
display(preflight["dependencies"])
display(preflight["workload"])
display(preflight["expected_outputs"])


,check,value
0,run_full_correction_validation,True
1,reuse_correction_prediction_files,False
2,alpha_rows,1011264
3,alpha_sites,10
4,alpha_loso_sites,"alpha_F, alpha_E, alpha_G"
5,beta_rows,280800
6,beta_sites,8
7,beta_date_window,2023-10-01 to 2024-09-30


,package,available,version
0,pandas,True,2.3.3
1,numpy,True,2.4.6
2,sklearn,True,1.9.0
3,xgboost,True,2.1.4
4,pyarrow,True,22.0.0
5,matplotlib,True,3.11.0


,step,count
0,Alpha LOSO m8_xgb training,3
1,Alpha LOSO m8_xgb prediction,3
2,Alpha LOSO m7_dtr batched inference,1
3,Alpha-to-Beta m8_xgb training,1
4,Beta m8_xgb prediction,1
5,Beta m7_dtr inference,1
6,Primary metric rows,16
7,Beta site metric rows,32


,artifact
0,01_correction_validation_plan.csv
1,01_correction_metrics.csv
2,02_correction_confusion_matrices.csv
3,table01_correction_metrics_summary.csv
4,table02_beta_transfer_key_metrics.csv
5,fig01a_confusion_matrices_day.png
6,fig01b_confusion_matrices_interval.png
7,fig02a_precision_recall_f1_day.png
8,fig02b_precision_recall_f1_interval.png
9,fig03_beta_site_precision_recall_f1_boxplot.png


## 3. Confirm Inputs, Folds, And Beta Site Order

The Alpha LOSO sites are recomputed from the current Alpha labels, so the notebook follows the data instead of a hard-coded site list. The Beta site order is also recomputed from current Beta labels.

In the full run, Beta predictions are made once per method across all Beta rows. Overall Beta metrics and per-site Beta metrics are then calculated downstream from those prediction frames, so the expensive model inference is not repeated just to make site-level tables or boxplots.

In [3]:
# Load final datasets once here for visible sanity checks before the workflow writes outputs.
alpha = h.load_dataset(article_root, cfg, "alpha")
beta = h.load_dataset(article_root, cfg, "beta")
alpha_sites = h.alpha_loso_sites(alpha, cfg)
print(f"Alpha LOSO sites: {alpha_sites}")
print(f"Beta rows: {len(beta):,}")
display(preflight["beta_rankings"][["substation_id", "rpf_days", "rpf_intervals", "rpf_day_pct"]])


Alpha LOSO sites: ['alpha_F', 'alpha_E', 'alpha_G']
Beta rows: 280,800


,substation_id,rpf_days,rpf_intervals,rpf_day_pct
1,beta_B,130,2496,35.519126
5,beta_F,118,2746,32.240437
6,beta_G,106,2478,28.961749
3,beta_D,96,1529,26.229508
4,beta_E,60,682,16.393443
0,beta_A,33,498,9.016393
7,beta_H,11,134,3.005464
2,beta_C,3,41,0.819672


## 4. Run Correction Validation

`h.run_correction_validation()` is the main workflow. In smoke mode it writes deterministic placeholder metrics and journal-style figures for layout checking. In full mode it performs exactly four `m8_xgb` trainings: three Alpha LOSO folds and one Alpha-to-Beta transfer model. Beta `m8_xgb` predictions are generated once for all sites, and Beta overall plus all-site metrics are calculated from that one prediction frame. Alpha `m7_dtr` inference is batched over the three held-out test folds to avoid repeated deterministic scans.

Public outputs use `day` and `interval` levels. The `interval` level is scoped to the configured daytime interval window, currently 06:00-18:00.


In [4]:
# Smoke mode writes placeholder outputs; full mode trains/evaluates the configured correction models.
result = h.run_correction_validation(article_root)
print(result["status"])
result["metrics"].head(20)



m7_threshold complete (0.3s):
  Site-days total:              1,095
  Skipped (missing data):       21
  Skipped (negative MW):        0
  Skipped (midday < 3 pts):     24
  Skipped (no candidates):      6
  Strict skip (no valid pair):  132
  Strict skip (threshold gate): 80
  Strict skip (daytime gate):   0
  Relaxed skip (no valid pair): 0
  Relaxed skip (daytime gate):  46
  Strict days flagged:          832
  Relaxed interval days:        998
  Relaxed-only days:            166
  Intervals flagged:            13,707
  Strict thresholds: min=5.00%, both=25.00%
XGB1 features: 220 columns, 9,580 rows (2,567 positive)
XGB2 features: 664 columns, 133,484 rows (33,875 positive)
XGB1 features: 212 columns, 365 rows (0 positive)
XGB2 features: 656 columns, 14,456 rows (0 positive)


c:\Users\samha\Documents\PyNRPF\publication\2_journal_article\notebooks\_experiment_helpers.py:1352: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["pred_interval"] = merged["pred_interval"].fillna(False).astype(bool).to_numpy()


XGB1 features: 220 columns, 9,580 rows (2,711 positive)
XGB2 features: 664 columns, 140,972 rows (38,015 positive)
XGB1 features: 212 columns, 365 rows (0 positive)
XGB2 features: 656 columns, 13,728 rows (0 positive)


c:\Users\samha\Documents\PyNRPF\publication\2_journal_article\notebooks\_experiment_helpers.py:1352: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["pred_interval"] = merged["pred_interval"].fillna(False).astype(bool).to_numpy()


XGB1 features: 220 columns, 9,580 rows (2,767 positive)
XGB2 features: 664 columns, 143,884 rows (38,423 positive)
XGB1 features: 212 columns, 365 rows (0 positive)
XGB2 features: 656 columns, 13,104 rows (0 positive)


c:\Users\samha\Documents\PyNRPF\publication\2_journal_article\notebooks\_experiment_helpers.py:1352: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["pred_interval"] = merged["pred_interval"].fillna(False).astype(bool).to_numpy()


XGB1 features: 221 columns, 10,643 rows (3,423 positive)
XGB2 features: 665 columns, 177,996 rows (47,980 positive)
XGB1 features: 219 columns, 2,928 rows (0 positive)
XGB2 features: 663 columns, 38,272 rows (0 positive)


c:\Users\samha\Documents\PyNRPF\publication\2_journal_article\notebooks\_experiment_helpers.py:1352: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["pred_interval"] = merged["pred_interval"].fillna(False).astype(bool).to_numpy()



m7_threshold complete (1.1s):
  Site-days total:              2,928
  Skipped (missing data):       401
  Skipped (negative MW):        0
  Skipped (midday < 3 pts):     2
  Skipped (no candidates):      80
  Strict skip (no valid pair):  740
  Strict skip (threshold gate): 1,021
  Strict skip (daytime gate):   1
  Relaxed skip (no valid pair): 0
  Relaxed skip (daytime gate):  240
  Strict days flagged:          683
  Relaxed interval days:        2,205
  Relaxed-only days:            1,522
  Intervals flagged:            18,518
  Strict thresholds: min=5.00%, both=25.00%
complete


,dataset,fold_id,method,level,support,positive_support,tp,fp,fn,tn,precision,recall,f1,is_placeholder,status
0,Alpha,alpha_holdout_alpha_F,m8_xgb,day,365,302,244,0,58,63,1.000000,0.807947,0.893773,False,complete
1,Alpha,alpha_holdout_alpha_F,m8_xgb,interval,18564,5587,3269,230,2318,12747,0.934267,0.585108,0.719569,False,complete
2,Alpha,alpha_holdout_alpha_F,m7_dtr,day,365,302,297,37,5,26,0.889222,0.983444,0.933962,False,complete
3,Alpha,alpha_holdout_alpha_F,m7_dtr,interval,18564,5587,4355,882,1232,12095,0.831583,0.779488,0.804693,False,complete
4,Alpha,alpha_holdout_alpha_E,m8_xgb,day,365,268,210,1,58,96,0.995261,0.783582,0.876827,False,complete
5,Alpha,alpha_holdout_alpha_E,m8_xgb,interval,18564,4127,3074,168,1053,14269,0.948180,0.744851,0.834306,False,complete
6,Alpha,alpha_holdout_alpha_E,m7_dtr,day,365,268,267,71,1,26,0.789941,0.996269,0.881188,False,complete
7,Alpha,alpha_holdout_alpha_E,m7_dtr,interval,18564,4127,3267,970,860,13467,0.771064,0.791616,0.781205,False,complete
8,Alpha,alpha_holdout_alpha_G,m8_xgb,day,365,251,213,4,38,110,0.981567,0.848606,0.910256,False,complete
9,Alpha,alpha_holdout_alpha_G,m8_xgb,interval,18564,4071,3145,181,926,14312,0.945580,0.772537,0.850345,False,complete


## 5. Inspect Transfer, Alpha Site, And Beta Site Rows

The primary metric CSV contains Alpha LOSO rows and Beta overall rows. `table01_correction_metrics_summary.csv` expands that view with the site identifier for every Alpha held-out fold and all Beta site rows. This table is the best first stop for checking whether a result is driven by a small number of difficult sites.

The boxplot figures summarise the site-level spread: Alpha uses 10 held-out-site rows per method/metric/level, and Beta uses 8 site rows per method/metric/level.

In [5]:
metrics = result["metrics"]
beta_transfer = metrics.loc[metrics["dataset"] == "Beta"].copy()
beta_site_metrics = result["beta_site_metrics"]

display(beta_transfer)
display(beta_site_metrics)


,dataset,fold_id,method,level,support,positive_support,tp,fp,fn,tn,precision,recall,f1,is_placeholder,status
12,Beta,beta_transfer,m8_xgb,day,2928,557,234,152,323,2219,0.606218,0.420108,0.496288,False,complete
13,Beta,beta_transfer,m8_xgb,interval,152100,10604,2608,1347,7996,140149,0.659418,0.245945,0.358266,False,complete
14,Beta,beta_transfer,m7_dtr,day,2928,557,424,1781,133,590,0.192290,0.761221,0.307024,False,complete
15,Beta,beta_transfer,m7_dtr,interval,152100,10604,6847,11671,3757,129825,0.369748,0.645700,0.470229,False,complete


,dataset,substation_id,fold_id,method,level,support,positive_support,tp,fp,fn,tn,precision,recall,f1,is_placeholder,status
0,Beta,beta_B,beta_site_beta_B,m8_xgb,day,366,130,11,1,119,235,0.916667,0.084615,0.154930,False,complete
1,Beta,beta_B,beta_site_beta_B,m8_xgb,interval,19032,2496,59,36,2437,16500,0.621053,0.023638,0.045542,False,complete
2,Beta,beta_B,beta_site_beta_B,m7_dtr,day,366,130,77,116,53,120,0.398964,0.592308,0.476780,False,complete
3,Beta,beta_B,beta_site_beta_B,m7_dtr,interval,19032,2496,1095,1006,1401,15530,0.521180,0.438702,0.476398,False,complete
4,Beta,beta_F,beta_site_beta_F,m8_xgb,day,366,118,49,18,69,230,0.731343,0.415254,0.529730,False,complete
5,Beta,beta_F,beta_site_beta_F,m8_xgb,interval,18980,2746,418,106,2328,16128,0.797710,0.152221,0.255657,False,complete
6,Beta,beta_F,beta_site_beta_F,m7_dtr,day,366,118,116,228,2,20,0.337209,0.983051,0.502165,False,complete
7,Beta,beta_F,beta_site_beta_F,m7_dtr,interval,18980,2746,2348,2075,398,14159,0.530861,0.855062,0.655043,False,complete
8,Beta,beta_G,beta_site_beta_G,m8_xgb,day,366,106,29,11,77,249,0.725000,0.273585,0.397260,False,complete
9,Beta,beta_G,beta_site_beta_G,m8_xgb,interval,19032,2478,573,154,1905,16400,0.788171,0.231235,0.357566,False,complete
